# 04f — Fine-tuning **score-first** (v2)

**Objetivo:** `LossSpec.finetune_score_primary()` — marcador 36 clases; W/D/L se deriva en inferencia.

- Datos: `DATA_ROOT` = `MyDrive/d10sformer`
- Código/ckpt: `PROJECT_ROOT` = `MyDrive/d10sformer-v2`
- Pretrain: `checkpoints_v1/pretrain_5ep/best.pt`


---
## 1. Setup

In [ ]:
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DRIVE = Path('/content/drive/MyDrive')

def _find_project_root() -> Path:
    for cand in (
        DRIVE / 'd10sformer-v2' / 'd10sformer-v2',
        DRIVE / 'd10sformer-v2',
    ):
        if (cand / 'src' / 'data' / 'collator.py').is_file():
            return cand.resolve()
    raise FileNotFoundError('No encuentro src/data/collator.py en d10sformer-v2')

if IN_COLAB:
    PROJECT_ROOT = _find_project_root()
    DATA_ROOT = (DRIVE / 'd10sformer').resolve()
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    DATA_ROOT = PROJECT_ROOT

# Un solo src en sys.path
_src = str(PROJECT_ROOT / 'src')
sys.path[:] = [p for p in sys.path if Path(p).name != 'src' and 'd10sformer' not in p]
sys.path.insert(0, _src)

from paths import ensure_paths, print_paths

paths = ensure_paths(project_root=PROJECT_ROOT, data_root=DATA_ROOT)
print_paths(paths)

# ── Auditoría de rutas críticas del 04f ─────────────────────────────
def _chk(label, p: Path) -> str:
    return f'{"OK" if p.exists() else "MISSING":7}  {label}: {p}'

checks = [
    _chk('collator.py', PROJECT_ROOT / 'src/data/collator.py'),
    _chk('vocabulary.py', PROJECT_ROOT / 'src/data/vocabulary.py'),
    _chk('trainer.py', PROJECT_ROOT / 'src/training/trainer.py'),
    _chk('vocab.json', paths.vocab_path),
    _chk('finetune_train.pkl', paths.corpus_dir / 'finetune_train.pkl'),
    _chk('pretrain best.pt', paths.checkpoints_v1 / 'pretrain_5ep/best.pt'),
]
print('\n--- Auditoría 04f ---')
print('\n'.join(checks))

_coll = (PROJECT_ROOT / 'src/data/collator.py').read_text(encoding='utf-8')
assert 'class LabelMappedCollator' in _coll, 'collator.py sin LabelMappedCollator — corré sync desde stg'
_tr = (PROJECT_ROOT / 'src/training/trainer.py').read_text(encoding='utf-8')
assert 'finetune_score_primary' in _tr, 'trainer.py sin finetune_score_primary'

ROOT = paths.project_root
DATA_PROCESSED = paths.data_processed
CORPUS_DIR = paths.corpus_dir
VOCAB_PATH = paths.vocab_path
CKPT_DIR = paths.checkpoints
CHECKPOINTS_V1 = paths.checkpoints_v1
DATA_RAW = paths.data_raw
DATA_INTERIM = paths.data_interim
print('\n✓ Setup OK')


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Necesitamos GPU. Runtime → Change runtime type → T4.')
print(f'torch: {torch.__version__}  |  GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from collections import Counter
import pandas as pd
from torch.utils.data import DataLoader

from data.dataset import MatchDataset
from models.d10sformer import D10Sformer, D10SformerConfig
from training.trainer import Trainer, TrainerConfig, LossSpec

# Checkpoint de pre-entrenamiento (v1 → mismo data_root)
PRETRAIN_CKPT = CHECKPOINTS_V1 / 'pretrain_5ep' / 'best.pt'
assert PRETRAIN_CKPT.exists(), f'No encuentro pretrain ckpt: {PRETRAIN_CKPT}'
print(f'✓ PRETRAIN_CKPT: {PRETRAIN_CKPT}')


In [ ]:
assert hasattr(LossSpec, 'finetune_score_primary'), 'Subí trainer.py v2 a Drive'
print('✓ trainer.py v2 OK')


---
## 2. Cargar corpus + computar distribución de clases

In [ ]:
import pickle
from data.vocabulary import FootballVocab
from data.tokenizer import MatchTokenizer

vocab = FootballVocab.load(VOCAB_PATH)
tokenizer = MatchTokenizer(vocab, max_seq_length=80)

# PAD: vocab en Drive usa [PAD] (no hace falta vocab.pad_id en vocabulary.py)
if hasattr(vocab, 'pad_id'):
    _pad_id = vocab.pad_id
else:
    _pad_id = vocab.token_to_id['[PAD]']
print(f'Vocab: {len(vocab):,} tokens  |  PAD={vocab.decode(_pad_id)!r} (id={_pad_id})')

with open(CORPUS_DIR / 'finetune_train.pkl', 'rb') as f:
    finetune_docs = pickle.load(f)
with open(CORPUS_DIR / 'val.pkl', 'rb') as f:
    val_docs = pickle.load(f)
with open(CORPUS_DIR / 'test.pkl', 'rb') as f:
    test_docs = pickle.load(f)

print(f'Fine-tune train: {len(finetune_docs):,}')
print(f'Val:             {len(val_docs):,}')
print(f'Test:            {len(test_docs):,}')


---
## 3. Datasets + Dataloaders (igual que 4d)

In [ ]:
from data.collator import MLMCollator, LabelMappedCollator

ds_train = MatchDataset(finetune_docs, tokenizer)
ds_val   = MatchDataset(val_docs, tokenizer)
ds_test  = MatchDataset(test_docs, tokenizer)

collator = LabelMappedCollator(MLMCollator(vocab, mlm_probability=0.15, seed=42))
_t = collator([ds_train[0]])
assert _t.score_labels.max() < 36

BATCH_SIZE = 64
train_loader = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator, num_workers=0)
val_loader   = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator, num_workers=0)
test_loader  = DataLoader(ds_test, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator, num_workers=0)
steps_per_epoch = len(train_loader)
print(f'Train batches/época: {steps_per_epoch}')


---
## 4. Modelo + Trainer con CLASS WEIGHTS

In [ ]:
# ── Distribución de clases de score en train + class weights ────────
# score_labels en el collator son locales (0-35, idx = h*6+a)
from collections import Counter

score_counts = Counter()
for sample in ds_train:
    local_id = collator.score_map.get(
        sample.target_score_id if sample.target_score_id is not None else -1, -1
    )
    if local_id >= 0:
        score_counts[local_id] += 1

n_total = sum(score_counts.values())
N_SCORE = 36
score_weights = [
    n_total / (N_SCORE * max(score_counts.get(i, 1), 1))
    for i in range(N_SCORE)
]

# Normalizar para que la media sea 1.0 (estabilidad numérica)
mean_w = sum(score_weights) / N_SCORE
score_weights = [w / mean_w for w in score_weights]

print(f'Total samples con score: {n_total}')
print('\nClases de EMPATE (draw scores):')
for h in range(6):
    idx = h * 6 + h
    print(f'  SCORE_{h}_{h} (idx {idx:2d}): count={score_counts.get(idx,0):4d}, weight={score_weights[idx]:.2f}')
print('\nClases más frecuentes (home wins):')
top = sorted(score_counts.items(), key=lambda x: -x[1])[:5]
for idx, cnt in top:
    h, a = divmod(idx, 6)
    print(f'  SCORE_{h}_{a} (idx {idx:2d}): count={cnt:4d}, weight={score_weights[idx]:.2f}')


In [ ]:
model_config = D10SformerConfig(
    vocab_size=len(vocab),
    d_model=256, num_layers=6, num_heads=8, d_ff=1024,
    max_seq_length=80, num_segments=8,
    dropout=0.1, attention_dropout=0.1,
    pad_token_id=_pad_id,
    tie_mlm_weights=True,
)
model = D10Sformer(model_config)

# Arrancamos limpios desde el pre-train (no desde el fine-tune anterior)
ckpt = torch.load(PRETRAIN_CKPT, map_location='cpu', weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
print(f'✓ Cargado pre-train (paso {ckpt["step"]}, val_loss={ckpt["best_val_loss"]:.4f})')

In [ ]:
EPOCHS = 15
MAX_STEPS = steps_per_epoch * EPOCHS
print(f'EPOCHS = {EPOCHS}  |  Pasos totales = {MAX_STEPS}')

trainer_config = TrainerConfig(
    lr=5e-5, weight_decay=0.01, grad_clip_norm=1.0,
    warmup_ratio=0.1,
    max_steps=MAX_STEPS,
    mixed_precision=True,
    log_every=25,
    eval_every=100,
    save_every=400,
    save_best=True,
    output_dir=str(CKPT_DIR),
    run_name='finetune_score_15ep',
    seed=42,
)
loss_spec = LossSpec.finetune_score_primary(
    score_class_weights=score_weights,
)
print(f'\nLossSpec: λ_mlm={loss_spec.lambda_mlm}, λ_score={loss_spec.lambda_score}, use_result={loss_spec.use_result}')

trainer = Trainer(
    model=model, train_loader=train_loader, val_loader=val_loader,
    config=trainer_config, loss_spec=loss_spec,
)
print(f'\nDevice: {trainer.device}  |  AMP: {trainer.use_amp}')
print('Score-first: sin result weights tensor')

---
## 5. Fine-tuning score-first (15 épocas)


In [ ]:
import time
t0 = time.time()
trainer.train()
elapsed = time.time() - t0
print(f'\n✓ Fine-tuning completado en {elapsed/60:.1f} min ({MAX_STEPS/elapsed:.1f} step/s)')
print(f'  Best val_loss: {trainer.best_val_loss:.4f}')

---
## 6. Evaluación detallada sobre VAL y TEST

In [ ]:
from eval.metrics import evaluate_all

best_path = trainer.output_dir / 'best.pt'
best_ckpt = torch.load(best_path, map_location=trainer.device, weights_only=False)
model.load_state_dict(best_ckpt['model_state_dict'])
model.eval()
print(f'✓ Cargado best (step={best_ckpt["step"]}, val_loss={best_ckpt["best_val_loss"]:.4f})')

# Índices de clases de score por resultado  (idx = h*6 + a, h/a ∈ 0..5)
_HOME_IDX = [h*6+a for h in range(6) for a in range(6) if h > a]   # 15 clases
_DRAW_IDX = [h*6+a for h in range(6) for a in range(6) if h == a]  #  6 clases
_AWAY_IDX = [h*6+a for h in range(6) for a in range(6) if h < a]   # 15 clases

@torch.no_grad()
def predict_result_probs(loader):
    """Deriva P(W/D/L) aggregando las 36 clases de score_logits.
    En score-primary, result_head NO se entrena → no usar result_logits.
    """
    all_probs, all_true = [], []
    for batch in loader:
        batch = batch.to(trainer.device)
        out = model(batch.token_ids, batch.segment_ids, attention_mask=batch.attention_mask)
        sp = F.softmax(out['score_logits'], dim=-1).cpu().numpy()  # (B, 36)
        labels = batch.result_labels.cpu().numpy()
        p_home = sp[:, _HOME_IDX].sum(axis=1)
        p_draw = sp[:, _DRAW_IDX].sum(axis=1)
        p_away = sp[:, _AWAY_IDX].sum(axis=1)
        probs  = np.stack([p_home, p_draw, p_away], axis=1)  # (B, 3)
        for p, y in zip(probs, labels):
            if y == -100: continue
            all_probs.append(p); all_true.append(int(y))
    return np.array(all_true), np.array(all_probs)

y_val,  p_val  = predict_result_probs(val_loader)
y_test, p_test = predict_result_probs(test_loader)

metrics_val  = evaluate_all(y_val,  p_val)
metrics_test = evaluate_all(y_test, p_test)
print('\n=== D10Sformer (4e, con class weights) ===')
print(f'{"métrica":<20} {"VAL":<12} {"TEST":<12}')
for k in ['log_loss', 'brier', 'ece', 'accuracy']:
    print(f'{k:<20} {metrics_val[k]:<12.4f} {metrics_test[k]:<12.4f}')

In [ ]:
# Confusion matrix: ¿ahora predice DRAW?
preds_val = p_val.argmax(axis=1)
cm = np.zeros((3, 3), dtype=int)
for yt, yp in zip(y_val, preds_val):
    cm[yt, yp] += 1

labels = ['home_win', 'draw', 'away_win']
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
for i in range(3):
    for j in range(3):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max() * 0.6 else 'black')
ax.set_xticks(range(3)); ax.set_xticklabels(labels)
ax.set_yticks(range(3)); ax.set_yticklabels(labels)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'Confusion matrix (VAL, n={len(y_val)}) — con class weights')
plt.colorbar(im); plt.tight_layout(); plt.show()

print(f'\nDistribución VAL: {Counter(y_val.tolist())}')
print(f'Predicciones VAL: {Counter(preds_val.tolist())}')
print(f'\nDraws predichos: {(preds_val == 1).sum()} de {len(y_val)} ({100*(preds_val == 1).sum()/len(y_val):.1f}%)')

---
## 7. Comparación final: baselines vs 4d (sin weights) vs 4e (con weights)

In [ ]:
# Baselines de Fase 1 (test)
baselines = {
    'LogReg':            {'log_loss': 0.8610, 'brier': 0.5071, 'ece': 0.0236, 'accuracy': 0.6005},
    'XGBoost':           {'log_loss': 0.8663, 'brier': 0.5105, 'ece': 0.0182, 'accuracy': 0.6026},
    'LightGBM':          {'log_loss': 0.8744, 'brier': 0.5135, 'ece': 0.0250, 'accuracy': 0.5990},
    'ELO solo':          {'log_loss': 1.0102, 'brier': 0.6101, 'ece': 0.0890, 'accuracy': 0.5510},
    'D10Sformer 4d (sin weights, test)':  {'log_loss': 0.8846, 'brier': 0.5221, 'ece': 0.0400, 'accuracy': 0.5912},
    'D10Sformer 4f (score-primary, val)':   {k: metrics_val[k]  for k in ['log_loss','brier','ece','accuracy']},
    'D10Sformer 4f (score-primary, test)':  {k: metrics_test[k] for k in ['log_loss','brier','ece','accuracy']},
}

import pandas as pd
df = pd.DataFrame(baselines).T
print(df.to_string(float_format=lambda x: f'{x:.4f}'))

best_baseline_ll  = min(baselines[k]['log_loss'] for k in ['LogReg','XGBoost','LightGBM'])
best_baseline_acc = max(baselines[k]['accuracy'] for k in ['LogReg','XGBoost','LightGBM'])
best_baseline_ece = min(baselines[k]['ece']      for k in ['LogReg','XGBoost','LightGBM'])

def cmp(v, ref, lower_better=True):
    if lower_better:
        return f'{"✓ GANA" if v < ref else "✗ pierde"} (Δ = {v - ref:+.4f})'
    return f'{"✓ GANA" if v > ref else "✗ pierde"} (Δ = {v - ref:+.4f})'

print(f'\n--- D10Sformer 4f score-primary (test) vs mejor baseline ---')
print(f'log_loss:  {metrics_test["log_loss"]:.4f}  vs  {best_baseline_ll:.4f}  →  {cmp(metrics_test["log_loss"], best_baseline_ll)}')
print(f'accuracy:  {metrics_test["accuracy"]:.4f}  vs  {best_baseline_acc:.4f}  →  {cmp(metrics_test["accuracy"], best_baseline_acc, lower_better=False)}')
print(f'ece:       {metrics_test["ece"]:.4f}  vs  {best_baseline_ece:.4f}  →  {cmp(metrics_test["ece"], best_baseline_ece)}')

---
## 8. Conclusiones de Fase 4f

Llenar al final:

- [ ] Draws predichos VAL: _____ de 293 reales (4d eran 12)
- [ ] D10Sformer 4f test log_loss: _____  vs LogReg 0.8610
- [ ] D10Sformer 4f test accuracy: _____  vs XGBoost 0.6026
- [ ] D10Sformer 4f test ECE: _____  vs XGBoost 0.0182
- [ ] ¿Ganamos en alguna métrica? _____
- [ ] ¿La confusion matrix se ve balanceada? _____

**Decisión próxima:**
- Si **ganamos en log_loss o ECE** → tenemos paper. Vamos a Fase 5 (eval profunda + reliability) y Fase 6 (Monte Carlo).
- Si seguimos perdiendo pero estamos cerca (Δ < 0.01) → ablations adicionales: ELO continuo en lugar de bucketizado, sin MLM aux.
- Si seguimos perdiendo por mucho → reportamos honestamente y usamos baselines para el Mundial.